# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LeylaAghayeva1/ml-search-engineering/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### ML Task Type

This project is a **ranking/scoring** problem.

The decision is not simply whether a page is declining, but **which pages should be reviewed first** by content editors. The model assigns each content page a priority score so that the highest-risk pages appear at the top of the review queue.

The people who use this output are content managers and SEO editors. They use the ranked list to decide where to spend their limited editing time.

A wrong prediction can either waste editor time by reviewing healthy pages or miss pages that actually need attention. Therefore, the model should rank the most important pages as highly as possible.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The starter dataset does not contain an observed future outcome suitable for supervised learning. Therefore, this notebook frames the problem as a ranking/scoring task, where a priority score is estimated from multiple content performance signals. In later stages, pages with the highest scores would be reviewed first by editors.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

priority_features = [
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]

df[priority_features].head()

,content_id,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update
0,content_304f48230142,3803,29,0.76,10.6,187,20
1,content_a1fb4e703a9e,15320,7,0.05,20.3,445,25
2,content_9aa793d4d895,12581,11,0.09,36.5,141,20
3,content_331d6c4de07b,11751,58,0.49,6.2,463,22
4,content_d99b7a2d90ca,19140,24,0.13,44.0,263,14


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success Metric

The primary evaluation metric is **Precision@50**.

Editors can only review a limited number of pages each day, so what matters most is how many truly declining pages appear in the first 50 recommendations.

A higher Precision@50 means that more editor effort is spent on pages that genuinely require attention.

Secondary metrics such as ROC-AUC or precision/recall may also be useful during experimentation, but Precision@50 best reflects the business decision.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Total content pages: {len(df):,}")
print(f"Unique clients: {df['client_id'].nunique()}")

display(
    df[
        [
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "content_age_days",
            "days_since_last_update"
        ]
    ].describe()
)

Total content pages: 30,000
Unique clients: 32


,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update
count,30000.000000,30000.000000,30000.000000,30000.00000,30000.00000,30000.000000
mean,5200.366300,16.097333,0.510733,16.34238,256.16780,46.098300
std,16838.019547,75.076958,3.279162,15.21679,132.70793,42.078709
min,1.000000,0.000000,0.000000,0.00000,90.00000,1.000000
25%,81.000000,0.000000,0.000000,6.20000,132.00000,20.000000
50%,731.000000,1.000000,0.070000,10.80000,236.00000,20.000000
75%,3615.250000,7.000000,0.290000,22.30000,333.00000,104.000000
max,517715.000000,4178.000000,100.000000,245.00000,564.00000,373.000000


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of Analysis

Each row represents **one content page** belonging to one client.

The dataframe contains performance signals such as impressions, clicks, CTR, average search position, content age, and time since the last update.

These measurements are used to estimate how likely the page is to require a content refresh.

The target column is **is_declining_label**, which indicates whether the page belongs to the declining group.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
unit_df = df[
    [
        "content_id",
        "client_id",
        "content_type",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "content_age_days",
        "days_since_last_update"
    ]
]

print("Shape:", unit_df.shape)

unit_df.head()

Shape: (30000, 9)


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,187,20
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,445,25
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,141,20
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,463,22
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,263,14


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML Instead of a Fixed Rule?

A simple rule such as "refresh pages with CTR below 1%" would ignore many important factors.

Page performance depends on several interacting signals including impressions, search position, content age, update history, click-through rate, and client differences. These relationships are difficult to capture using a single hand-written rule.

Machine learning can combine many signals simultaneously and learn patterns that are not obvious to humans.

The output supports decision-making by helping editors prioritize the pages most likely to benefit from a refresh rather than relying on one threshold.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
simple_rule = df[df["ctr"] < 1]

print(f"Pages selected by the simple CTR rule: {len(simple_rule)}")

simple_rule[
    [
        "content_id",
        "ctr",
        "impressions_90d",
        "avg_position",
        "content_age_days",
        "days_since_last_update"
    ]
].head()

comparison_columns = [
    "ctr",
    "impressions_90d",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]

display(simple_rule[comparison_columns].describe())

Pages selected by the simple CTR rule: 28291


,ctr,impressions_90d,avg_position,content_age_days,days_since_last_update
count,28291.000000,28291.000000,28291.000000,28291.000000,28291.000000
mean,0.151639,5306.018592,16.698798,257.097169,46.505108
std,0.214861,17086.067275,15.389614,133.368786,41.982967
min,0.000000,1.000000,0.000000,90.000000,1.000000
25%,0.000000,96.000000,6.400000,133.000000,20.000000
50%,0.050000,785.000000,11.100000,236.000000,20.000000
75%,0.240000,3732.500000,22.900000,336.500000,104.000000
max,0.990000,517715.000000,245.000000,564.000000,373.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.